# Profile ray-traced channel snapshots at random locations

**[FACT — scope]** This notebook calls the production `SionnaRTChannelBackend.compute_snapshot()` for **FAU, POWDER, AERPAW, and LOS_empty**, at every integer **maximum depth from 3 to 15**. Each plotted point is the **arithmetic mean of 100 snapshots**, each at a different random TX/RX pair. This replaces the earlier fixed-position median experiment.

**[FACT — implementation]** One snapshot is the complete complex FIR tap vector for one scene, TX/RX pair, and radio configuration—not one individual tap. The backend reloads the scene on **every call**.

```text
100 random TX/RX pairs per scene ── reuse at every depth
                                       │
start clock → load scene → configure nodes → trace paths → transfer taps → synchronize → stop clock
                                       │
                       sum of 100 durations / 100 → mean (ms)
```

**[FACT — boundary]** Timing includes scene reload, node setup, ray tracing, tap conversion, and final synchronization. It excludes imports, bounds discovery, random-position generation, configuration construction, warm-ups, validation, file writes, plotting, visualization, GNU Radio's `set_taps()`, and streaming scheduler latency. This is **taps-ready latency**, not input-to-output sample latency or pure solver time.

Source: [backend.py](../python/rt_channel_emulation/backend.py), [runtime.py](../python/rt_channel_emulation/runtime.py).


## 1. Select the project Python environment

**[FACT — procedure]** First install the block using [README](../README.md). From the repository root, install the optional notebook requirements:

```sh
.venv/bin/python -m pip install -e ".[profiling]"
```

Open this notebook in your notebook editor, select this repository's **.venv Python**, and run the cells in order. The extra provides Matplotlib (plot), IPython kernel (notebook execution), and NBClient (optional headless execution); it does not add requirements to the block's normal installation.

**[FACT — CPU requirement]** If Dr.Jit cannot find LLVM, configure the path **before importing Sionna**. The next cell provides an editable example, not an automatic system modification. Restart the kernel after changing the environment. See [project LLVM instructions](../README.md) and [IPython kernel instructions](https://ipython.readthedocs.io/en/stable/install/kernel_install.html).


In [ ]:
import csv
from dataclasses import asdict, replace
from datetime import datetime, timezone
from importlib.metadata import version
import inspect
import json
import os
from pathlib import Path
import platform
from statistics import mean
import sys
from time import perf_counter_ns

# Uncomment ONLY if needed; replace with the LLVM library on your machine.
os.environ["DRJIT_LIBLLVM_PATH"] = "/opt/local/libexec/llvm-18/lib/libLLVM.dylib"

import numpy as np
import drjit as dr
import mitsuba as mi
from sionna.rt import PathSolver, load_scene
from rt_channel_emulation.backend import (
    SimulationConfig, SionnaRTChannelBackend, resolve_scene_path,
)
import rt_channel_emulation.backend as backend_module

# Render plots inside the notebook rather than launching a separate Qt window.
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

# Notebook editors commonly start the kernel in either the repo or tests/.
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "python/rt_channel_emulation/backend.py").is_file():
    raise RuntimeError("Start the notebook from the repository root or tests/.")

print("Python:", sys.executable)
print("Backend:", backend_module.__file__)
print("Mitsuba variant:", mi.variant())
print("Dr.Jit CPU threads:", dr.thread_count())


jitc_llvm_init(): LLVM API initialization failed ..


ImportError: jit_init_thread_state(): the LLVM backend is inactive because the LLVM shared library ("libLLVM.dylib") could not be found! Set the DRJIT_LIBLLVM_PATH environment variable to specify its path.

## 2. Choose settings and sample node locations

**[FACT — definitions]** Maximum depth limits propagation interactions along a path; it is not the FIR tap count. At 1 MS/s, sampled delay spacing is 1 microsecond; lags -6 through 32 give **39 taps**. Propagation flags, solver budgets, carrier frequency, antennas, and tap support remain unchanged. See [Sionna PathSolver](https://nvlabs.github.io/sionna/rt/api/paths_solvers.html) and [backend.py](../python/rt_channel_emulation/backend.py).

**[FACT — sampling design]** Independently sample TX and RX uniformly in each scene's axis-aligned XY geometry bounds, at **z = 2 m in the scene coordinate frame**. This is not terrain-relative height. The empty scene has no geometry bounds, so it uses FAU's bounds. Sampling is not collision-checked: positions inside buildings are allowed, and valid all-zero channels are retained. The random seed and all coordinates are saved.

**[FACT — controlled comparison]** Generate 100 pairs once per scene and reuse them, in the same order, at all 13 depths. Each pair contributes exactly one measured snapshot per depth: **4 × 13 × 100 = 5,200 measured snapshots**. FAU and Empty use identical coordinates because both bounds and position seed match. Two additional warm-up calls per scene/depth use the first two pairs and are excluded from the mean.

AERPAW uses its full geometry bounds, including its 5.4 km × 5.4 km ground plane; this is not a building-only sampling area. Earlier saved three-scene results do not contain AERPAW timings.

**[INFERENCE — high confidence]** Matching positions across depths reduces location-related differences between points. Two warm-ups reduce some first-use compilation effects but cannot guarantee every execution branch is warm. This design samples spatial variation, not repeated-call variability at a single location. Fixed seeds do not guarantee bitwise-identical results across software versions/devices.

Sources: [NumPy uniform sampling](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.uniform.html), [Sionna scene loader](https://nvlabs.github.io/sionna/rt/api/scene.html).


In [ ]:
MAX_DEPTHS = list(range(3, 16))  # 3, 4, ..., 15 (inclusive).
SNAPSHOTS = 100               # One measured snapshot per random TX/RX pair.
WARMUPS = 2                   # Additional discarded calls per scene/depth.
POSITION_SEED = 2026          # Position RNG; separate from the ray solver seed.
NODE_HEIGHT_M = 2.0           # Fixed scene z-coordinate, not height above terrain.
MACHINE_LABEL = platform.processor() or platform.machine()  # Edit to name your CPU/GPU.

# Shared radio settings. Template positions are replaced by sampled pairs below.
SCENES = {
    name: SimulationConfig(
        scene_path=alias, tx_position=(0, 0, NODE_HEIGHT_M),
        rx_position=(1, 0, NODE_HEIGHT_M), sample_rate=1e6,
        carrier_frequency_hz=2.4e9, l_min=-6, l_max=32, seed=42, visualize=False,
    )
    for name, alias in (("FAU", "FAU_scene"), ("POWDER", "POWDER"),
                        ("AERPAW", "AERPAW_scene"), ("Empty", "LOS_empty"))
}

print(f"{len(SCENES) * len(MAX_DEPTHS)} points; "
      f"{SNAPSHOTS} measured + {WARMUPS} warm-up snapshots per point.")


In [ ]:
def sample_positions(xy_bounds, count, height_m, seed):
    """Return positions shaped (count, 2 nodes, 3 coordinates), in metres.

    xy_bounds = [[xmin, ymin], [xmax, ymax]]. Node 0 is TX; node 1 is RX.
    Each x/y coordinate is independently uniform; z is the fixed height_m.
    No building/terrain rejection is applied. The seed makes draws repeatable.
    """
    bounds = np.asarray(xy_bounds, dtype=float)
    if (bounds.shape != (2, 2) or not np.all(np.isfinite(bounds))
            or np.any(bounds[1] <= bounds[0])):
        raise ValueError("Use finite XY bounds with positive width and height.")
    if count < 1 or not np.isfinite(height_m):
        raise ValueError("Use a positive position count and a finite node height.")
    rng = np.random.default_rng(seed)
    positions = np.empty((count, 2, 3))
    positions[:, :, :2] = rng.uniform(bounds[0], bounds[1], size=(count, 2, 2))
    positions[:, :, 2] = height_m
    return positions


In [ ]:
# Discover world-space geometry bounds once, OUTSIDE the timed snapshots.
# The production backend still reloads the scene inside each measured call.
scene_bounds = {}
for name in SCENES:
    if name == "Empty":
        continue  # Its sampling domain is assigned from FAU below.
    scene = load_scene(resolve_scene_path(SCENES[name].scene_path), merge_shapes=True)
    bbox = scene.mi_scene.bbox()
    if not bbox.valid():
        raise ValueError(f"{name} has no valid geometry bounds.")
    scene_bounds[name] = np.asarray([bbox.min, bbox.max], dtype=float)[:, :2].tolist()
    del scene

scene_bounds["Empty"] = scene_bounds["FAU"]  # Explicit domain for the unbounded empty scene.
positions = {
    name: sample_positions(bounds, SNAPSHOTS, NODE_HEIGHT_M, POSITION_SEED)
    for name, bounds in scene_bounds.items()
}
for name, bounds in scene_bounds.items():
    print(f"{name}: XY bounds {bounds}; {len(positions[name])} TX/RX pairs; z={NODE_HEIGHT_M} m")


## 3. Define the measurement

**[FACT — calculation]** For pair i, `elapsed_ms[i] = (end_ns - start_ns) / 1_000_000`. The reported **arithmetic mean** is `sum(elapsed_ms) / 100`. It is not a median or a mean of medians. Illustrative example: **[9, 10, 11, 12, 80] ms → mean 24.4 ms**. Every valid measured call is included, including slow calls and all-zero channels.

**[FACT — clock and completion]** Python's monotonic performance counter measures elapsed time, including waiting. Dr.Jit can launch deferred/asynchronous work, so the notebook synchronizes before starting and before stopping the clock. The final wait is included. Invalid taps or solver exceptions abort the sweep rather than entering the mean as fast successes.

Sources: [Python performance counter](https://docs.python.org/3/library/time.html#time.perf_counter_ns), [Python arithmetic mean](https://docs.python.org/3/library/statistics.html#statistics.mean), [Dr.Jit synchronization](https://drjit.readthedocs.io/en/stable/reference.html#drjit.sync_thread).


In [ ]:
def profile_snapshot(backend, config, positions, *, warmups):
    """Measure one snapshot per TX/RX pair; return raw times and their mean.

    Inputs: production backend, radio/depth config, positions shaped (N, 2, 3)
    in metres, and a non-negative warm-up count. Output: result dictionary,
    including per-pair milliseconds and the arithmetic mean over all N pairs.
    Warm-ups reuse the first pairs; their durations are not recorded.
    """
    positions = np.asarray(positions, dtype=float)
    if (positions.ndim != 3 or positions.shape[1:] != (2, 3)
            or len(positions) < 1 or not np.all(np.isfinite(positions)) or warmups < 0):
        raise ValueError("Use finite positions shaped (N >= 1, 2, 3) and warmups >= 0.")
    if config.visualize:
        raise ValueError("Disable visualization for this taps-ready benchmark.")

    # Construct and validate configurations before timing any trace.
    configs = [replace(config, tx_position=tuple(tx), rx_position=tuple(rx))
               for tx, rx in positions]
    for index in range(warmups):
        backend.compute_snapshot(configs[index % len(configs)])
        dr.sync_thread()

    samples = []
    expected_taps = config.l_max - config.l_min + 1
    for pair_id, pair_config in enumerate(configs, start=1):
        dr.sync_thread()  # Exclude pending work from the preceding call.
        start_ns = perf_counter_ns()
        snapshot = backend.compute_snapshot(pair_config)
        dr.sync_thread()  # Include completion, not just submission.
        elapsed_ms = (perf_counter_ns() - start_ns) / 1_000_000

        # Validation is outside the timed interval. has_paths is not a path count.
        if len(snapshot.taps) != expected_taps or not np.all(np.isfinite(snapshot.taps)):
            raise ValueError("Snapshot has the wrong tap count or non-finite taps.")
        samples.append({"pair_id": pair_id, "elapsed_ms": elapsed_ms,
                        "has_nonzero_taps": bool(snapshot.has_paths)})

    return {
        "scene_alias": config.scene_path,
        "max_depth": config.max_depth,
        "mean_ms": mean(sample["elapsed_ms"] for sample in samples),
        "samples": samples,
        "snapshot_count": len(samples),
        "tap_count": expected_taps,
        "nonzero_tap_runs": sum(sample["has_nonzero_taps"] for sample in samples),
    }


## 4. Run the sweep

**[FACT — procedure]** Stop other flowgraphs, viewers, and heavy workloads first. The loop runs serially, at depths 3 through 15, without clearing caches or replacing the production scene loader. Progress lines report a measured **mean**; `nonzero taps` counts snapshots with a nonzero tap vector, not propagation paths. Raw times and summaries are flushed to disk after each completed scene/depth point, outside the timer.

**[INFERENCE — high confidence]** Background load, CPU/GPU frequency, thermal throttling, and cache history can affect timing. One ascending sweep does not remove order effects. Repeat the experiment before generalizing performance. Deep traces may take substantial time and memory; interrupt the kernel to stop. Completed points remain on disk, but the current unfinished batch is not saved.


In [ ]:
results = []
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = ROOT / "tests" / "profiling_results" / run_stamp
output_dir.mkdir(parents=True, exist_ok=False)  # Preserve every previous run.

# Record actual environment, configuration templates, bounds, and sampled positions.
# pair_id in the raw CSV is a 1-based index into positions_m[scene].
metadata = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "status": "running",
    "statistic": "arithmetic_mean",
    "machine_label": MACHINE_LABEL,
    "platform": platform.platform(),
    "logical_cpu_count": os.cpu_count(),
    "drjit_cpu_threads": dr.thread_count(),
    "mitsuba_variant": mi.variant(),
    "python": sys.version,
    "python_executable": sys.executable,
    "backend_source": str(Path(backend_module.__file__).resolve()),
    "versions": {name: version(name) for name in
                 ("rt-channel-emulation", "sionna-rt", "drjit", "mitsuba", "numpy", "matplotlib")},
    "solver_signature": str(inspect.signature(PathSolver.__call__)),
    "scene_config_templates": {name: asdict(config) for name, config in SCENES.items()},
    "max_depths": MAX_DEPTHS,
    "warmups_per_point": WARMUPS,
    "snapshots_per_point": SNAPSHOTS,
    "position_seed": POSITION_SEED,
    "node_height_m": NODE_HEIGHT_M,
    "xy_bounds_m": scene_bounds,
    "positions_m": {name: pairs.tolist() for name, pairs in positions.items()},
    "sampling": "Independent uniform TX/RX XY; fixed scene z; no collision rejection; "
                "same pairs/order at all depths; Empty uses FAU bounds and pairs.",
    "timing_scope": "compute_snapshot + final sync; includes scene reload; excludes set_taps/streaming",
}
with (output_dir / "metadata.json").open("w") as handle:
    json.dump(metadata, handle, indent=2)
print("Run directory:", output_dir, flush=True)

raw_fields = ["scene", "max_depth", "pair_id", "elapsed_ms", "has_nonzero_taps"]
summary_fields = ["scene", "scene_alias", "max_depth", "mean_ms", "snapshot_count",
                  "tap_count", "nonzero_tap_runs"]
backend = SionnaRTChannelBackend()
try:
    with (output_dir / "snapshot_times.csv").open("w", newline="") as raw_file, \
         (output_dir / "snapshot_means.csv").open("w", newline="") as summary_file:
        raw_writer = csv.DictWriter(raw_file, fieldnames=raw_fields)
        summary_writer = csv.DictWriter(summary_file, fieldnames=summary_fields,
                                        extrasaction="ignore")
        raw_writer.writeheader()
        summary_writer.writeheader()
        for scene_name, base_config in SCENES.items():
            for depth in MAX_DEPTHS:
                config = replace(base_config, max_depth=depth)
                row = profile_snapshot(backend, config, positions[scene_name], warmups=WARMUPS)
                row["scene"] = scene_name
                results.append(row)
                raw_writer.writerows({"scene": scene_name, "max_depth": depth, **sample}
                                     for sample in row["samples"])
                summary_writer.writerow(row)
                raw_file.flush()
                summary_file.flush()
                print(f"{scene_name:6s} depth={depth:2d}: mean {row['mean_ms']:9.3f} ms "
                      f"(nonzero taps: {row['nonzero_tap_runs']}/{row['snapshot_count']})",
                      flush=True)
    metadata["status"] = "complete"
finally:
    backend.close()  # Clean up on success, solver error, or interruption.
    metadata["finished_utc"] = datetime.now(timezone.utc).isoformat()
    metadata["completed_points"] = len(results)
    if metadata["status"] != "complete":
        metadata["status"] = "incomplete"
    with (output_dir / "metadata.json").open("w") as handle:
        json.dump(metadata, handle, indent=2)


## 5. Compare mean snapshot latency

**[FACT — axes]** X = maximum depth; Y = arithmetic mean wall-clock milliseconds over **100 complete snapshots at random TX/RX locations**. Each scene has a distinct color and marker. The plot rejects missing, duplicate, or undersampled points. This is a spatial average for the chosen sampling domain—not a median or a worst-case deadline.


In [ ]:
expected = {(name, depth) for name in SCENES for depth in MAX_DEPTHS}
actual = {(row["scene"], row["max_depth"]) for row in results}
if (actual != expected or len(results) != len(expected)
        or any(row["snapshot_count"] != SNAPSHOTS for row in results)):
    raise ValueError("Sweep incomplete: finish all requested snapshots before plotting.")

colors = {"FAU": "tab:blue", "POWDER": "tab:orange",
          "AERPAW": "tab:purple", "Empty": "tab:green"}
markers = {"FAU": "o", "POWDER": "s", "AERPAW": "D", "Empty": "^"}
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
for scene_name in SCENES:
    rows = sorted((row for row in results if row["scene"] == scene_name),
                  key=lambda row: row["max_depth"])
    ax.plot([row["max_depth"] for row in rows],
            [row["mean_ms"] for row in rows],
            color=colors[scene_name], marker=markers[scene_name],
            label=scene_name, linewidth=2)

ax.set(
    xlabel="Maximum path depth",
    ylabel="Mean snapshot wall-clock time (ms)",
    title=f"Taps-ready latency ({SNAPSHOTS} random TX/RX pairs per scene, reused at every depth)",
    xticks=MAX_DEPTHS,
    ylim=(0, None),
)
ax.grid(True, alpha=0.25)
ax.legend(title="Scene")
plt.show()


## 6. Save the plot

**[FACT — output]** The sweep already saved raw per-pair timings (`snapshot_times.csv`), mean summaries (`snapshot_means.csv`), and environment/configuration/coordinate metadata (`metadata.json`) in a new timestamped directory. The next cell saves `snapshot_latency.png` there. Raw CSV `pair_id` is the 1-based index into that scene's `positions_m` list in metadata; each entry is `[TX_xyz, RX_xyz]` in metres. Warm-up calls are not in the CSV.

**[FACT — reproducibility]** Previous fixed-position median results remain untouched in their own directories. Generated results are ignored by Git; archive the selected complete run, code, and scene assets before using the figure in a paper. Do not compare old fixed-pair medians with new random-pair means as if only the software changed.


In [ ]:
fig.savefig(output_dir / "snapshot_latency.png", dpi=180)
print("Saved measurements:", output_dir)


## Conclusion: interpret only the measurements

**[FACT — scope]** Each point is the measured arithmetic mean of 100 **taps-ready latencies**, including scene reload, over the recorded random TX/RX pairs. At depth 15 the output still has 39 taps; depth does not guarantee physical accuracy. Empty-scene time includes loading, setup, solver dispatch, and tap conversion. Source: [backend.py](../python/rt_channel_emulation/backend.py).

**[INFERENCE — high confidence]** If a point's mean were 200 ms, its reciprocal would correspond to about 5 serial snapshots/s (`1000 / 200`) under that workload, not a guaranteed update deadline. Slow calls, application overhead, and competing work still matter. This illustrative calculation does not measure whether the configured FIR processes samples at 1 MS/s.

**[FACT — limitations]** This experiment uses one spatial sample set per scene, not repeated timings at every pair. Uniform XY sampling includes potential building interiors, and z is fixed rather than terrain-following. The experiment does not measure RF accuracy, continuous mobility, visualization cost, or GNU Radio tap-application latency. Report the hardware, solver settings, position sampling, and scene bounds with the measured means.
